# Notebook 02 — Backtesting Analysis

10 years of SPY data. Rolling 252-day Historical VaR backtested annually.

Key finding demonstrated:
- 2008/2020 can fail Christoffersen (violations cluster) even while passing Kupiec
- Basel traffic light zones by year
- Violation clustering is the most dangerous failure mode in practice


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

from src.var_models import historical_var
from src.backtesting import kupiec_test, christoffersen_test, basel_traffic_light, full_backtest_report

print('Libraries loaded.')

In [ ]:
# 10 years of SPY
spy = yf.download('SPY', start='2014-01-01', end='2024-01-01', progress=False)['Close']
returns = spy.pct_change().dropna()
print(f'SPY: {len(returns)} observations ({returns.index[0].date()} to {returns.index[-1].date()})')

## 1. Rolling 252-Day Historical VaR

In [ ]:
confidence_level = 0.05  # 95% VaR
window = 252

rolling_var = returns.rolling(window).quantile(confidence_level)
violations = returns < rolling_var

fig = go.Figure()
fig.add_trace(go.Scatter(x=returns.index, y=returns * 100,
                          name='SPY Returns (%)', line=dict(color='steelblue', width=0.5)))
fig.add_trace(go.Scatter(x=rolling_var.index, y=rolling_var * 100,
                          name='Rolling 95% VaR', line=dict(color='orange', width=2)))

viol_dates = returns.index[violations]
viol_vals  = returns[violations] * 100
fig.add_trace(go.Scatter(x=viol_dates, y=viol_vals, mode='markers',
                          name='Violations', marker=dict(color='red', size=5)))

fig.update_layout(
    title='SPY Rolling 95% VaR Backtest (2014-2024)',
    xaxis_title='Date', yaxis_title='Return (%)',
    height=500
)
fig.show()
print(f'Total violations: {violations.sum()} / {len(violations)} ({violations.mean():.2%})')

## 2. Annual Backtesting Results

In [ ]:
annual_results = []

for year in range(2015, 2024):
    year_returns = returns[str(year)]
    # Use prior-year data to set VaR
    prior_year_returns = returns[str(year - 1)]
    if len(prior_year_returns) < 100 or len(year_returns) < 50:
        continue
    
    var_value = historical_var(prior_year_returns, confidence_level)
    var_series = pd.Series(var_value, index=year_returns.index)
    
    kupiec = kupiec_test(year_returns, var_series, confidence_level)
    cc     = christoffersen_test(year_returns, var_series, confidence_level)
    traffic = basel_traffic_light(kupiec['n_violations'], kupiec['n_observations'])
    
    annual_results.append({
        'Year':            year,
        'Observations':    kupiec['n_observations'],
        'Violations':      kupiec['n_violations'],
        'Expected':        round(kupiec['expected_violations'], 1),
        'Kupiec p-value':  round(kupiec['p_value'], 4),
        'CC p-value':      round(cc['p_value_cc'], 4),
        'Ind p-value':     round(cc['p_value_ind'], 4),
        'n11 (clusters)':  cc['n11'],
        'Zone':            traffic['zone'].upper(),
        'Passes Kupiec':   '✓' if kupiec['passes'] else '✗',
        'Passes CC':       '✓' if cc['passes_cc'] else '✗',
    })

annual_df = pd.DataFrame(annual_results)
print(annual_df.to_string(index=False))

## 3. Violation Clustering Visualisation

In [ ]:
# Show consecutive violation patterns
violations_series = (returns < rolling_var).astype(int)
consecutive = (violations_series == 1) & (violations_series.shift(1) == 1)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=violations_series.index, y=violations_series.cumsum(),
    name='Cumulative Violations', line=dict(color='red')
))

# Mark years
for year in [2020]:
    fig.add_vrect(
        x0=f'{year}-01-01', x1=f'{year}-12-31',
        fillcolor='rgba(255,0,0,0.1)', line_width=0,
        annotation_text=str(year), annotation_position='top left'
    )

fig.update_layout(
    title='Cumulative VaR Violations — Clustering in Stress Periods',
    xaxis_title='Date', yaxis_title='Cumulative Violations',
    height=400
)
fig.show()

print(f'Consecutive violation pairs (n11): {consecutive.sum()}')
print('These represent periods where the model failed on back-to-back days,')
print('exactly what Christoffersen independence test detects.')

## 4. Basel Traffic Light Summary

In [ ]:
zone_colors = {'GREEN': 'green', 'YELLOW': 'orange', 'RED': 'red'}

print('\nBasel III Traffic Light by Year:')
print('-' * 50)
for _, row in annual_df.iterrows():
    zone = row['Zone']
    symbol = {'GREEN': '🟢', 'YELLOW': '🟡', 'RED': '🔴'}.get(zone, '⚪')
    print(f"{int(row['Year'])}  {symbol} {zone:<8}  "
          f"Violations: {int(row['Violations'])}/{int(row['Expected'])}  "
          f"Kupiec: {row['Passes Kupiec']}  CC: {row['Passes CC']}")